# Lezione 35 — Inferenza con Gemma

> **Modulo:** Transformer e modello open · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 33 (generazione), Lezione 34 (KerasHub).
>
> Obiettivo pratico unico: generare testo con **Gemma** via `.generate()`,
> collegando temperatura/sampler alla Lezione 33 — e costruire nel progetto un
> estrattore con **fallback a regole** che funziona anche senza modello.
>
> ⚠️ **Nota di ambiente**: come la Lezione 34, le celle Gemma sono guardate e
> saltate qui; il codice e' l'API reale. Vedi `course/research_gaps.md`.

## Teoria essenziale

Caricato il task model (Lezione 34), l'inferenza e' una sola chiamata:
`gemma.generate(prompt, max_length=...)`. Il *come* si sceglie il prossimo
token — greedy o sampling con temperatura — e' la **stessa** meccanica della
Lezione 33: in KerasHub si configura con `compile(sampler=...)` (es. un
`GreedySampler` o un sampler con temperatura). Il modello sostituisce il bigram
come stimatore della distribuzione del prossimo token; il ciclo autoregressivo
e' identico.

In [1]:
# Guardia di ambiente: KerasHub + pesi Gemma sono un extra opzionale (`ml`)
# e richiedono download autenticato di un modello di diversi GB. In questo
# ambiente non sono presenti, quindi le celle che usano il modello vengono
# SALTATE in modo pulito (il notebook resta eseguibile in CI). Con GPU e
# accesso a Gemma configurato, la stessa cella gira davvero.
GEMMA_AVAILABLE = False
_motivo = ""
try:
    import keras  # noqa: F401
    import keras_hub  # noqa: F401
    GEMMA_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    _motivo = f"{type(exc).__name__}: {exc}"

if GEMMA_AVAILABLE:
    print("KerasHub disponibile: le celle del modello verranno eseguite.")
else:
    print("KerasHub/Gemma NON disponibile in questo ambiente.")
    print("Motivo:", _motivo or "pacchetto assente")
    print("Le celle che richiedono il modello stampano [saltato]; "
          "il resto della lezione gira comunque.")

KerasHub/Gemma NON disponibile in questo ambiente.
Motivo: ModuleNotFoundError: No module named 'keras'
Le celle che richiedono il modello stampano [saltato]; il resto della lezione gira comunque.


In [2]:
if GEMMA_AVAILABLE:
    import keras_hub
    gemma = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")
    # sampler = come scegliere il prossimo token (cfr. Lezione 33)
    gemma.compile(sampler="greedy")
    prompt = "Extract the city from: Marco visited Glasgow with his son.\nCity:"
    print(gemma.generate(prompt, max_length=32))
else:
    print("[saltato] gemma.generate(prompt, max_length=32)")
    print("Con il modello, questo prompt estrarrebbe la citta' dal testo.")

[saltato] gemma.generate(prompt, max_length=32)
Con il modello, questo prompt estrarrebbe la citta' dal testo.


## Il progetto: Memory AI Lab, passo 35 — estrazione con fallback

Il progetto deve estrarre entita' (citta', persone) dalle memorie. Con Gemma si
farebbe via prompt; senza, si usa un **fallback a regole** (l'euristica delle
maiuscole della Lezione 26). Progetto un'unica funzione che usa il modello se
disponibile e altrimenti le regole — cosi' il sistema funziona sempre.

In [3]:
import re

def _estrai_a_regole(testo):
    cand = re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo))
    return [c for c in cand if c not in {"The", "A"}]

def estrai_entita(testo, usa_modello=GEMMA_AVAILABLE):
    if usa_modello and GEMMA_AVAILABLE:
        prompt = f"List the proper nouns in: {testo}\nAnswer:"
        risposta = gemma.generate(prompt, max_length=48)  # noqa: F821
        return [w.strip() for w in risposta.split("Answer:")[-1].split(",") if w.strip()]
    return _estrai_a_regole(testo)

# controllo di non-regressione: il fallback funziona sempre
ent = estrai_entita("Marco visited Glasgow with his son", usa_modello=False)
assert "Marco" in ent and "Glasgow" in ent, ent
print("entita' estratte (fallback a regole):", ent)

entita' estratte (fallback a regole): ['Marco', 'Glasgow']


## Riepilogo

1. Inferenza = `gemma.generate(prompt, max_length=...)`.
2. Il sampler (`compile(sampler=...)`) e' il greedy/temperatura della Lezione 33.
3. Il modello sostituisce il bigram; il **ciclo autoregressivo** e' lo stesso.
4. Un buon sistema ha un **fallback** (regole) quando il modello non c'e'.
5. L'estrazione via prompt e' potente ma va sempre validata (Lezioni 36–37).

## Quiz

1. Dove ritrovi la temperatura della Lezione 33 nell'API di Gemma?
2. Perche' e' utile un fallback a regole?
3. Perche' l'output di un prompt di estrazione va comunque validato?

*(Risposte: 1. nel sampler configurato con `compile`; 2. per far funzionare il
sistema senza GPU/modello e come rete di sicurezza; 3. perche' il modello puo'
allucinare o cambiare formato — serve un validatore, Lezione 36.)*